In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

#Reload data
df = pd.read_csv('../data/raw/nab/machine_temperature_system_failure.csv', parse_dates=['timestamp'], index_col='timestamp')

#Reload anomaly windows
with open('../data/raw/nab/combined_windows.json') as f:
    windows = json.load(f)

anomaly_windows = windows['realKnownCause/machine_temperature_system_failure.csv']
print(df.shape)
print('Data loaded successfully')

(22695, 1)
Data loaded successfully


In [18]:
def extract_window_fatures(window):
    return{
        'mean': np.mean(window),
        'std': np.std(window),
        'min': np.min(window),
        'max': np.max(window),
        'range': np.max(window) - np.min(window),
        'kurtosis': pd.Series(window).kurtosis(),
        'skewness': pd.Series(window).skew()
    }

WINDOW_SIZE = 50

feature_rows = []
indices = []
values = df['value'].values

for i in range(WINDOW_SIZE, len(values)):
    window = values[i - WINDOW_SIZE:i]
    features = extract_window_fatures(window)
    feature_rows.append(features)
    indices.append(df.index[i])

features_df = pd.DataFrame(feature_rows, index = indices)

print(features_df.shape)
print(features_df.head())

(22645, 7)
                          mean       std        min        max      range  \
2013-12-03 01:25:00  81.307319  2.239710  73.967322  85.344624  11.377302   
2013-12-03 01:30:00  81.521412  2.029624  74.935882  85.344624  10.408742   
2013-12-03 01:35:00  81.728749  1.869484  76.124162  85.344624   9.220463   
2013-12-03 01:40:00  81.893250  1.725404  78.140707  85.344624   7.203917   
2013-12-03 01:45:00  82.021553  1.679498  78.710418  85.344624   6.634206   

                     kurtosis  skewness  
2013-12-03 01:25:00  2.297913 -1.158348  
2013-12-03 01:30:00  1.622272 -0.814675  
2013-12-03 01:35:00  0.523965 -0.314788  
2013-12-03 01:40:00 -0.513213  0.082851  
2013-12-03 01:45:00 -0.713270  0.165262  


In [20]:
labels = pd.Series(0, index=features_df.index)

for window in anomaly_windows:
    start = pd.Timestamp(window[0])
    end = pd.Timestamp(window[1])
    labels[(labels.index >= start) & (labels.index <= end)] = 1

print(f'Normal: {(labels == 0).sum()}')
print(f'Anomalous: {(labels == 1).sum()}')

Normal: 20377
Anomalous: 2268
